# Drug Discovery Sim Env — GRPO Training (Colab)

End-to-end notebook: clones the repo, installs deps, starts the OpenEnv server in the background, runs a random baseline, and then GRPO-trains Qwen2.5-3B-Instruct on live rollouts.

Hardware: free **T4** is enough; **A100** recommended for full throughput.

## 1. Setup

In [ ]:
!nvidia-smi || echo 'no GPU'

In [ ]:
import os, subprocess
REPO_URL = os.environ.get('REPO_URL', 'https://huggingface.co/spaces/VasuBB/drug-discovery-sim-env')
if not os.path.exists('drug-discovery-sim-env'):
    subprocess.run(['git', 'clone', REPO_URL, 'drug-discovery-sim-env'], check=True)
%cd drug-discovery-sim-env

In [ ]:
!pip install -q -r server/requirements.txt
!pip install -q 'transformers>=4.45' 'datasets>=2.20' 'accelerate>=0.34' 'trl>=0.12' trackio matplotlib
# Optional: !pip install -q unsloth

## 2. Start the env server in the background

In [ ]:
import subprocess, time, requests
proc = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for _ in range(40):
    try:
        r = requests.get('http://localhost:8000/health', timeout=1)
        if r.ok:
            print('server up:', r.json())
            break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError('server never came up')

## 3. Random-policy baseline (no GPU needed)

In [ ]:
!python training/train_model.py --baseline --baseline-episodes 20 --base-url http://localhost:8000

In [ ]:
import json
baseline = json.loads(open('outputs/baseline/baseline_summaries.json').read())
rewards = [b['total_reward'] for b in baseline]
print(f'baseline mean reward: {sum(rewards)/len(rewards):.3f}  (n={len(rewards)})')

## 4. GRPO training

In [ ]:
!python training/train_model.py \
    --base-url http://localhost:8000 \
    --model Qwen/Qwen2.5-3B-Instruct \
    --output-dir outputs/grpo \
    --epochs 1 \
    --no-unsloth

## 5. Plot reward curves

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

rewards = []
log_files = sorted(glob.glob('outputs/grpo/checkpoint-*/trainer_state.json'))
if log_files:
    state = json.load(open(log_files[-1]))
    for entry in state.get('log_history', []):
        if 'reward' in entry:
            rewards.append(entry['reward'])
if not rewards:
    print('no reward log found; train longer or check trainer_state.json path')
else:
    os.makedirs('plots', exist_ok=True)
    plt.figure(figsize=(7, 4))
    plt.plot(rewards, label='train reward')
    plt.xlabel('training step')
    plt.ylabel('mean episode reward')
    plt.title('GRPO training reward — Drug Discovery Sim Env')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig('plots/reward_curve.png', dpi=140)
    plt.show()
    print('saved plots/reward_curve.png')

In [ ]:
proc.terminate()